In [1]:
import json
from dotenv import load_dotenv
from openai.types.chat import ChatCompletionMessage, ChatCompletionToolParam

from tool_runner import run_tool
from memory import add_messages, get_messages, save_tool_response, AIMessage

load_dotenv()

True

In [2]:
import aisuite as ai

client = ai.Client()

In [3]:
MAX_AI_TURNS = 10

def run_llm(messages: list[AIMessage], tools: list[ChatCompletionToolParam], model: str = "openai:gpt-4.1") -> ChatCompletionMessage:
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        tools=tools, # <-- Your list of tools with get_current_time
    )

    return response.choices[0].message


def run_agent(user_message: str, tools: list[ChatCompletionToolParam]) -> list[AIMessage]:
    turns = 1
    add_messages([{"role": "user", "content": user_message}])

    while True:
        history: list[AIMessage] = get_messages()
        if turns > MAX_AI_TURNS:
            print(f"Reached max AI Conversation limit of {MAX_AI_TURNS} turns")
            return history
        
        response: ChatCompletionMessage = run_llm(history, tools)
        add_messages([response])

        print(json.dumps(response.model_dump(), indent=2, default=str))

        # Process all tool calls before checking for a final answer
        if response.tool_calls:
            for tool_call in response.tool_calls:
                tool_result = run_tool(tool_call, user_message)
                save_tool_response(tool_call.id, tool_result)
            turns += 1
            continue

        # No tool calls remaining — LLM has produced a final response
        return get_messages()


In [4]:
user_message: str = input("Enter your question here:")
tools: list[ChatCompletionToolParam] = [
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": "Returns the current local time as a formatted string (HH:MM:SS).",
            "parameters": {}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_location_coordinates_from_ip",
            "description": "Resolves the user's public IP address to geographic coordinates (latitude and longitude).",
            "parameters": {}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather_from_ip",
            "description": "Returns the current, high, and low temperature in Fahrenheit based on the user's IP-derived location.",
            "parameters": {}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "write_text_file",
            "description": "Creates or overwrites a .txt file at the given path with the provided text content.",
            "parameters": {
                "type": "object",
                "properties": {
                    "file_path": {
                        "type": "string",
                        "description": "Relative or absolute path for the .txt file, e.g. 'notes/summary.txt'"
                    },
                    "content": {
                        "type": "string",
                        "description": "The full text to write into the file"
                    }
                },
                "required": ["file_path", "content"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "generate_qr_code",
            "description": "Generates a styled QR code PNG image encoding the given text or URL, with an embedded image in the center.",
            "parameters": {
                "type": "object",
                "properties": {
                    "data": {
                        "type": "string",
                        "description": "Text or URL to encode in the QR code"
                    },
                    "filename": {
                        "type": "string",
                        "description": "Name for the output PNG file without extension, e.g. 'my-qr'"
                    },
                    "image_path": {
                        "type": "string",
                        "description": "Path to an image to embed in the center of the QR code"
                    }
                },
                "required": ["data", "filename", "image_path"],
                "additionalProperties": False
            }
        }
    }
]

run_agent(user_message, tools)

{
  "content": null,
  "refusal": null,
  "role": "assistant",
  "annotations": [],
  "audio": null,
  "function_call": null,
  "tool_calls": [
    {
      "id": "call_oGffrXXzyF116bNsUiTyjFTV",
      "function": {
        "arguments": "{\"data\": \"www.deeplearning.com\", \"filename\": \"deeplearning-qr\", \"image_path\": \"dl_logo.jpg\"}",
        "name": "generate_qr_code"
      },
      "type": "function"
    },
    {
      "id": "call_WGxTbKVUQaqOFSS5Tq1eddfO",
      "function": {
        "arguments": "{}",
        "name": "get_weather_from_ip"
      },
      "type": "function"
    }
  ]
}
{
  "content": null,
  "refusal": null,
  "role": "assistant",
  "annotations": [],
  "audio": null,
  "function_call": null,
  "tool_calls": [
    {
      "id": "call_5rPINeeHaZOR2JMR2kN4jFri",
      "function": {
        "arguments": "{\"file_path\":\"weather_note.txt\",\"content\":\"Current weather: 28.0\u00b0F\\nHigh: 33.5\u00b0F\\nLow: 21.4\u00b0F\"}",
        "name": "write_text_file"
    

[{'role': 'user',
  'content': 'Can you help me create a qr code that goes to www.deeplearning.com from the image dl_logo.jpg? Also write me a txt note with the current weather please.'},
 {'content': None,
  'refusal': None,
  'role': 'assistant',
  'annotations': [],
  'audio': None,
  'function_call': None,
  'tool_calls': [{'id': 'call_oGffrXXzyF116bNsUiTyjFTV',
    'function': {'arguments': '{"data": "www.deeplearning.com", "filename": "deeplearning-qr", "image_path": "dl_logo.jpg"}',
     'name': 'generate_qr_code'},
    'type': 'function'},
   {'id': 'call_WGxTbKVUQaqOFSS5Tq1eddfO',
    'function': {'arguments': '{}', 'name': 'get_weather_from_ip'},
    'type': 'function'}]},
 {'role': 'tool',
  'tool_call_id': 'call_oGffrXXzyF116bNsUiTyjFTV',
  'content': 'QR code saved as deeplearning-qr.png containing: www.deeplearning.com...'},
 {'role': 'tool',
  'tool_call_id': 'call_WGxTbKVUQaqOFSS5Tq1eddfO',
  'content': 'Current: 28.0°F, High: 33.5°F, Low: 21.4°F'},
 {'content': None,
 